In [ ]:
#| default_exp renderers.chatgpt_image

# renderers.chatgpt_image

> Renderer for the OpenAI image generation model (DALL-E 3 / gpt-image-1).
>
> Supports a single reference image per call.
> API key: `OPENAI_API_KEY` environment variable.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import base64
import os
from pathlib import Path

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
class ChatGPTImageRenderer(BaseRenderer):
    """Image generation via OpenAI image models (DALL-E 3 / gpt-image-1).

    Supports a single reference image input per call for character consistency.
    No LoRA, no negative prompt support.

    Requires: OPENAI_API_KEY environment variable.
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        super().__init__(model_spec, config)
        self._model_cfg = config.chatgpt_image

    def _client(self):
        from openai import OpenAI  # type: ignore
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise EnvironmentError("OPENAI_API_KEY is not set")
        return OpenAI(api_key=api_key)

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> RenderResult:
        import asyncio
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg, reference_images
        )

    def _render_sync(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None,
    ) -> RenderResult:
        import httpx
        client = self._client()
        prompt = panel.visual_prompt
        w, h = output_cfg.resolved_dimensions()

        # Pick first character's reference image if available
        ref_image_b64: str | None = None
        if reference_images:
            for char_name in panel.characters_present:
                ref_path = reference_images.get(char_name)
                if ref_path and ref_path.exists():
                    ref_image_b64 = base64.b64encode(ref_path.read_bytes()).decode()
                    break  # only one reference image supported

        # gpt-image-1 supports image input; dall-e-3 does not
        if ref_image_b64 and self._model_cfg.model == "gpt-image-1":
            response = client.images.edit(
                model=self._model_cfg.model,
                image=base64.b64decode(ref_image_b64),
                prompt=prompt,
                size=f"{w}x{h}",
                response_format="url",
            )
        else:
            response = client.images.generate(
                model=self._model_cfg.model,
                prompt=prompt,
                size=f"{w}x{h}",
                quality=self._model_cfg.quality,
                response_format="url",
                n=1,
            )

        image_url = response.data[0].url
        image_bytes = httpx.get(image_url).content

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        out_path.write_bytes(image_bytes)

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=prompt,
            metadata={"model": self._model_cfg.model, "ref_image_used": ref_image_b64 is not None},
        )

In [ ]:
# Construction test (no API call)
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig
from manhualizer.renderers.chatgpt_image import ChatGPTImageRenderer

renderer = ChatGPTImageRenderer(MODELS["chatgpt-image"], RendererConfig())
assert renderer.model_spec.capabilities.reference_images
assert not renderer.model_spec.capabilities.multi_image_input
assert not renderer.model_spec.capabilities.lora
print("ChatGPTImageRenderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()